In [ ]:
# In[1]:

import pandas as pd

# Reuse variables where possible
metric_file = "metric.csv"
trace_file = "trace.csv"
log_file = "log.csv"
error_file = "error_logs.csv"

# Load files
df_metric = pd.read_csv(metric_file)
df_trace = pd.read_csv(trace_file)
df_log = pd.read_csv(log_file)
df_error = pd.read_csv(error_file)

# Helper to parse timestamp column into timezone-aware UTC datetime column 'ts'
def ensure_ts(df, ts_col='timestamp'):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df[ts_col], unit='s', utc=True)
    return df

df_metric = ensure_ts(df_metric)
df_trace = ensure_ts(df_trace)
df_log = ensure_ts(df_log)
df_error = ensure_ts(df_error)

# Helper to build compact summary for a file
def summarize(df, namecol=None, sample_cols=None):
    # Unique cmdb_id as a Series (compact)
    unique_cmdb = pd.Series(sorted(df['cmdb_id'].dropna().unique()), name='unique_cmdb_id')
    # Unique names for KPI/feature/log (if provided)
    if namecol and namecol in df.columns:
        unique_names = pd.Series(sorted(df[namecol].dropna().unique()), name=f'unique_{namecol}')
        # counts per name (top 20)
        counts = (
            df.groupby(namecol)
              .size()
              .reset_index(name='count')
              .sort_values('count', ascending=False)
              .head(20)
              .reset_index(drop=True)
        )
    else:
        unique_names = pd.Series([], name='unique_names', dtype=object)
        counts = pd.DataFrame(columns=['name','count'])
    # earliest and latest timestamps as ISO UTC strings
    if 'ts' in df.columns and not df['ts'].isna().all():
        earliest = df['ts'].min().isoformat()
        latest = df['ts'].max().isoformat()
    else:
        earliest = None
        latest = None
    # sample rows (up to 5) - include ts and sample columns if provided
    if sample_cols:
        cols = [c for c in sample_cols if c in df.columns]
        sample_df = df.sort_values('ts').head(5)[cols + ['ts']]
    else:
        # default small sample
        sample_df = df.sort_values('ts').head(5)[['cmdb_id','ts'] + [c for c in df.columns if c not in ('cmdb_id','timestamp','ts')][:3]]
    # Compact summary as a dict-like object (but keep DataFrames/Series for display)
    summary = {
        'unique_cmdb': unique_cmdb,
        'unique_names': unique_names,
        'top_counts': counts,
        'earliest_ts': earliest,
        'latest_ts': latest,
        'samples': sample_df
    }
    return summary

# Build summaries for each file with appropriate name columns and sample columns
summary_metric = summarize(df_metric, namecol='kpi_name', sample_cols=['cmdb_id','kpi_name','value'])
summary_trace  = summarize(df_trace,  namecol='trace_name', sample_cols=['cmdb_id','trace_name','value'])
summary_log    = summarize(df_log,    namecol='log_name',   sample_cols=['cmdb_id','log_name','value'])
# error_logs: no kpi/log_name column; use 'message' for unique names/counts and sample columns
summary_error  = summarize(df_error,  namecol='message',    sample_cols=['cmdb_id','message'])

# Display the four summaries (compact)
summary_metric, summary_trace, summary_log, summary_error

```
Out[1]:
```
```python
# Build a plain-English summary string using the previously computed summaries
um = summary_metric['unique_cmdb'].tolist()
uk = summary_metric['unique_names'].tolist()
tc = summary_metric['top_counts'].head(7)
samp = summary_metric['samples'].head(5)

um_t = summary_trace['unique_cmdb'].tolist()
ut = summary_trace['unique_names'].tolist()
ttc = summary_trace['top_counts'].head(7)
tsamp = summary_trace['samples'].head(5)

ul = summary_log['unique_cmdb'].tolist()
ukl = summary_log['unique_names'].tolist()
ltc = summary_log['top_counts'].head(7)
lsamp = summary_log['samples'].head(5)

# Compose compact representations for top-count entries and samples
def top_counts_to_str(df, name_col):
    if df.empty:
        return "none"
    return "; ".join([f"{row[name_col]}({int(row['count'])})" for _, row in df.iterrows()])

def samples_to_str(df, name_col):
    if df.empty:
        return "none"
    lines = []
    for _, r in df.iterrows():
        name = r.get(name_col, "")
        lines.append(f"{r['cmdb_id']} | {name} | {r.get('value','')} | {r['ts'].isoformat()}")
    return " ; ".join(lines)

metric_top = top_counts_to_str(tc, 'kpi_name')
metric_samples = samples_to_str(samp, 'kpi_name')

trace_top = top_counts_to_str(ttc, 'trace_name')
trace_samples = samples_to_str(tsamp, 'trace_name')

log_top = top_counts_to_str(ltc, 'log_name')
log_samples = samples_to_str(lsamp, 'log_name')

summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    f"- Unique cmdb_id ({len(um)}): {', '.join(um)}\n"
    f"- Unique kpi_name ({len(uk)}): {', '.join(uk)}\n"
    f"- Top KPI counts (top entries): {metric_top}\n"
    f"- Time range: {summary_metric['earliest_ts']} to {summary_metric['latest_ts']}\n"
    f"- Sample rows (up to 5): {metric_samples}\n\n"
    "trace.csv:\n"
    f"- Unique cmdb_id ({len(um_t)}): {', '.join(um_t)}\n"
    f"- Unique trace_name ({len(ut)}): {', '.join(ut[:10]) + (' ...' if len(ut)>10 else '')}\n"
    f"- Top trace feature counts (top entries): {trace_top}\n"
    f"- Time range: {summary_trace['earliest_ts']} to {summary_trace['latest_ts']}\n"
    f"- Sample rows (up to 5): {trace_samples}\n\n"
    "log.csv:\n"
    f"- Unique cmdb_id ({len(ul)}): {', '.join(ul)}\n"
    f"- Unique log_name ({len(ukl)}): {', '.join(ukl)}\n"
    f"- Top log counts: {log_top}\n"
    f"- Time range: {summary_log['earliest_ts']} to {summary_log['latest_ts']}\n"
    f"- Sample rows (up to 5): {log_samples}\n\n"
    "error_logs.csv:\n"
    "- This file appears empty in the loaded data (no cmdb_id, message, counts, or timestamps present).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

({'unique_cmdb': 0                 adservice
1               cartservice
2           checkoutservice
3           currencyservice
4              emailservice
5                  frontend
6         frontend-external
7            paymentservice
8     productcatalogservice
9     recommendationservice
10                    redis
11          shippingservice
Name: unique_cmdb_id, dtype: object, 'unique_names': 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: unique_kpi_name, dtype: object, 'top_counts':      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    100
7       error     75, 'earliest_ts': '2024-01-16T10:07:00+00:00', 'latest_ts': '2024-01-16T10:31:00+00:00', 'samples':                   cmdb_id    kpi_name         value                        ts
0               adservice         cpu  6.952937e-01 2024-01-16 10:07:00+00:00
51  productcatalogservice         mem  1.044854e+07 2024-01-16 10:07:00+00:00
50  productcatalogservice  latency-90  4.422039e-03 2024-01-16 10:07:00+00:00
49  productcatalogservice  latency-50  1.955335e-03 2024-01-16 10:07:00+00:00
48  productcatalogservice         cpu  3.513565e+00 2024-01-16 10:07:00+00:00}, {'unique_cmdb': 0          checkoutservice
1          currencyservice
2             emailservice
3          frontendservice
4           paymentservice
5    productcatalogservice
6    recommendationservice
7                     root
Name: unique_cmdb_id, dtype: object, 'unique_names': 0         trace.from_checkoutservice.duration_mean
1          trace.from_checkoutservice.duration_p95
2            trace.from_checkoutservice.error_rate
3             trace.from_checkoutservice.row_count
4         trace.from_frontendservice.duration_mean
                          ...                     
39        trace.to_productcatalogservice.row_count
40    trace.to_recommendationservice.duration_mean
41     trace.to_recommendationservice.duration_p95
42       trace.to_recommendationservice.error_rate
43        trace.to_recommendationservice.row_count
Name: unique_trace_name, Length: 44, dtype: object, 'top_counts':                                       trace_name  count
0                      trace.from_root.row_count    175
1                     trace.from_root.error_rate    175
2                   trace.from_root.duration_p95    175
3                  trace.from_root.duration_mean    175
4       trace.from_frontendservice.duration_mean    125
5        trace.from_frontendservice.duration_p95    125
6          trace.from_frontendservice.error_rate    125
7           trace.from_frontendservice.row_count    125
8       trace.from_checkoutservice.duration_mean    125
9        trace.from_checkoutservice.duration_p95    125
10         trace.from_checkoutservice.error_rate    125
11          trace.from_checkoutservice.row_count    125
12  trace.to_productcatalogservice.duration_mean    100
13   trace.to_productcatalogservice.duration_p95    100
14     trace.to_productcatalogservice.error_rate    100
15      trace.to_productcatalogservice.row_count    100
16      trace.to_recommendationservice.row_count     75
17     trace.to_recommendationservice.error_rate     75
18   trace.to_recommendationservice.duration_p95     75
19  trace.to_recommendationservice.duration_mean     75, 'earliest_ts': '2024-01-16T10:07:00+00:00', 'latest_ts': '2024-01-16T10:31:00+00:00', 'samples':                    cmdb_id                                     trace_name       value                        ts
0          checkoutservice       trace.from_checkoutservice.duration_mean    0.003768 2024-01-16 10:07:00+00:00
97   productcatalogservice  trace.from_recommendationservice.duration_p95    0.000030 2024-01-16 10:07:00+00:00
98   productcatalogservice    trace.from_recommendationservice.error_rate    0.000000 2024-01-16 10:07:00+00:00
99   productcatalogservice     trace.from_recommendationservice.row_count  207.000000 2024-01-16 10:07:00+00:00
100  productcatalogservice                  trace.from_root.duration_mean    0.002897 2024-01-16 10:07:00+00:00}, {'unique_cmdb': 0                adservice
1              cartservice
2          checkoutservice
3          currencyservice
4             emailservice
5                 frontend
6           paymentservice
7    recommendationservice
8                    redis
9          shippingservice
Name: unique_cmdb_id, dtype: object, 'unique_names': 0    log.error_count
1      log.row_count
Name: unique_log_name, dtype: object, 'top_counts':           log_name  count
0  log.error_count    229
1    log.row_count    229, 'earliest_ts': '2024-01-16T10:07:00+00:00', 'latest_ts': '2024-01-16T10:31:00+00:00', 'samples':                   cmdb_id         log_name  value                        ts
0               adservice  log.error_count      0 2024-01-16 10:07:00+00:00
17        shippingservice    log.row_count    150 2024-01-16 10:07:00+00:00
16        shippingservice  log.error_count      0 2024-01-16 10:07:00+00:00
15  recommendationservice    log.row_count    207 2024-01-16 10:07:00+00:00
14  recommendationservice  log.error_count      0 2024-01-16 10:07:00+00:00}, {'unique_cmdb': Series([], Name: unique_cmdb_id, dtype: object), 'unique_names': Series([], Name: unique_message, dtype: object), 'top_counts': Empty DataFrame
Columns: [message, count]
Index: [], 'earliest_ts': None, 'latest_ts': None, 'samples': Empty DataFrame
Columns: [cmdb_id, message, ts]
Index: []})```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is timezone-aware UTC
if 'ts' not in df_metric.columns:
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # ensure dtype
    df_metric['ts'] = pd.to_datetime(df_metric['ts'], utc=True)

# Restrict to the specified components
components = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]
df_m_sub = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Compute global (full-series for these components) group statistics BEFORE incident-window filtering
group_keys = ['cmdb_id', 'kpi_name']

agg_funcs = {
    'count': ('value', 'size'),
    'min': ('value', 'min'),
    'max': ('value', 'max'),
    'p50': ('value', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan),
    'p90': ('value', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna())>0 else np.nan),
    'p95': ('value', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan),
}

# Use groupby.agg with named aggregations
summary = df_m_sub.groupby(group_keys).agg(
    count=('value', 'size'),
    min=('value', 'min'),
    max=('value', 'max'),
    p50=('value', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan),
    p90=('value', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna())>0 else np.nan),
    p95=('value', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan),
).reset_index()

# Parse incident window (UTC)
window_start = pd.to_datetime("2024-01-16 10:04:37", utc=True)
window_end   = pd.to_datetime("2024-01-16 10:34:37", utc=True)

# Filter to incident-window subset
df_window = df_m_sub[(df_m_sub['ts'] >= window_start) & (df_m_sub['ts'] <= window_end)].copy()

# Compute count_in_window per group
count_in_window = df_window.groupby(group_keys).agg(count_in_window=('value','size')).reset_index()

# Merge window rows with p95 thresholds to compute exceed flags
if not df_window.empty:
    df_win_with_p95 = df_window.merge(summary[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
    df_win_with_p95['exceed'] = df_win_with_p95['value'] >= df_win_with_p95['p95']
    # exceed counts per group
    exceed_counts = df_win_with_p95.groupby(group_keys).agg(
        exceed_count_in_window=('exceed','sum')
    ).reset_index()
    # earliest exceed timestamp per group
    exceed_times = (
        df_win_with_p95[df_win_with_p95['exceed']]
        .groupby(group_keys)
        .agg(earliest_exceed_ts=('ts','min'))
        .reset_index()
    )
    # convert earliest_exceed_ts to ISO strings, keep NaT as None later
    if not exceed_times.empty:
        exceed_times['earliest_exceed_ts'] = exceed_times['earliest_exceed_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        # format timezone as +00:00
        exceed_times['earliest_exceed_ts'] = exceed_times['earliest_exceed_ts'].str.replace('+0000', '+00:00')
else:
    exceed_counts = pd.DataFrame(columns=group_keys + ['exceed_count_in_window'])
    exceed_times = pd.DataFrame(columns=group_keys + ['earliest_exceed_ts'])

# Merge counts and exceed info into the summary table
result = summary.merge(count_in_window, on=group_keys, how='left')
result = result.merge(exceed_counts, on=group_keys, how='left')
result = result.merge(exceed_times, on=group_keys, how='left')

# Fill NaNs for groups with no window rows
result['count_in_window'] = result['count_in_window'].fillna(0).astype(int)
result['exceed_count_in_window'] = result['exceed_count_in_window'].fillna(0).astype(int)
# For earliest_exceed_ts, keep NaN -> convert to None
result['earliest_exceed_ts'] = result['earliest_exceed_ts'].where(result['earliest_exceed_ts'].notna(), None)

# Reorder/select columns as requested
result_df = result[['cmdb_id','kpi_name','count','min','max','p50','p90','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts']]

# Sample up to 5 metric rows from the incident-window filtered subset
sample_rows = df_window.sort_values('ts').loc[:, ['cmdb_id','kpi_name','value','ts']].head(5)

# Display the compact results
result_df, sample_rows

```
Out[2]:
```
# Build a plain-English summary string from the previously computed results
import pandas as pd
# Reuse result_df and sample_rows from the kernel state

# Basic counts and components
num_groups = int(result_df.shape[0])
components = sorted(result_df['cmdb_id'].unique().tolist())
window_start = "2024-01-16T10:04:37+00:00"
window_end   = "2024-01-16T10:34:37+00:00"

# Exceedance overview
groups_with_exceed = int((result_df['exceed_count_in_window'] > 0).sum())
full_exceed_groups = result_df[result_df['exceed_count_in_window'] == result_df['count_in_window']]
num_full_exceed = int(full_exceed_groups.shape[0])

# Distribution of exceed counts (compact)
exceed_dist = result_df['exceed_count_in_window'].value_counts().sort_index().to_dict()

# Earliest exceed timestamps (top 5 earliest)
earliest_exceeds = (
    result_df[result_df['earliest_exceed_ts'].notna()]
    .sort_values('earliest_exceed_ts')
    .loc[:, ['cmdb_id','kpi_name','earliest_exceed_ts']]
    .head(5)
)
earliest_list = [
    f"{r['cmdb_id']} | {r['kpi_name']} | {r['earliest_exceed_ts']}"
    for _, r in earliest_exceeds.iterrows()
]

# Top problematic KPI by full-window exceed (if any)
full_exceed_list = full_exceed_groups[['cmdb_id','kpi_name']].apply(lambda r: f"{r['cmdb_id']}:{r['kpi_name']}", axis=1).tolist()

# Compose sample rows (up to 5) from the incident-window subset
sample_lines = []
for _, r in sample_rows.iterrows():
    sample_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} | value={r['value']:.6g} | ts={r['ts'].isoformat()}")

summary = (
    f"Telemetry summary for selected components within window {window_start} to {window_end}:\n\n"
    f"- Component-KPI groups analyzed: {num_groups}\n"
    f"- Components included: {', '.join(components)}\n"
    f"- Groups with at least one value >= global P95 in the window: {groups_with_exceed}\n"
    f"- Groups where every window row met/ exceeded P95 (count_in_window == exceed_count_in_window): {num_full_exceed}\n"
    f"- Distribution of exceed_count_in_window (value: number of groups): {exceed_dist}\n"
    f"- Example earliest exceed timestamps (up to 5):\n  " + ("\n  ".join(earliest_list) if earliest_list else "none") + "\n\n"
    f"- KPI groups with full-window exceed (if any): " + (", ".join(full_exceed_list) if full_exceed_list else "none") + "\n\n"
    f"Up to 5 sample metric rows from the incident-window subset:\n  " + ("\n  ".join(sample_lines) if sample_lines else "none")
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  count           min           max           p50           p90           p95  count_in_window  exceed_count_in_window         earliest_exceed_ts
0         checkoutservice         cpu     25  2.873450e-01  5.334684e-01  3.692888e-01  4.268142e-01  4.305833e-01               25                       2  2024-01-16T10:13:00+00:00
1         checkoutservice  latency-50     25  6.967891e-02  1.477790e-01  1.197250e-01  1.377830e-01  1.387241e-01               25                       2  2024-01-16T10:14:00+00:00
2         checkoutservice  latency-90     25  2.084292e-01  2.761190e-01  2.297555e-01  2.616647e-01  2.699161e-01               25                       2  2024-01-16T10:12:00+00:00
3         checkoutservice         mem     25  1.145679e+07  1.196005e+07  1.174917e+07  1.187103e+07  1.191515e+07               25                       2  2024-01-16T10:20:00+00:00
4         checkoutservice      socket     25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00               25                      25  2024-01-16T10:07:00+00:00
..                    ...         ...    ...           ...           ...           ...           ...           ...              ...                     ...                        ...
28  recommendationservice  latency-50     25  7.399720e-03  2.523531e-01  7.587655e-03  2.513819e-01  2.516226e-01               25                       2  2024-01-16T10:23:00+00:00
29  recommendationservice  latency-90     25  9.541374e-03  4.521552e-01  4.761123e-02  4.504918e-01  4.510485e-01               25                       2  2024-01-16T10:25:00+00:00
30  recommendationservice         mem     25  4.597139e+07  4.629804e+07  4.619824e+07  4.625590e+07  4.627874e+07               25                       2  2024-01-16T10:21:00+00:00
31  recommendationservice      socket     25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00               25                      25  2024-01-16T10:07:00+00:00
32  recommendationservice    workload     25  1.655435e+01  2.017568e+01  1.825905e+01  1.886137e+01  1.903874e+01               25                       2  2024-01-16T10:08:00+00:00

[33 rows x 11 columns],                   cmdb_id    kpi_name         value                        ts
14        checkoutservice         cpu  3.874473e-01 2024-01-16 10:07:00+00:00
60  recommendationservice    workload  1.782865e+01 2024-01-16 10:07:00+00:00
59  recommendationservice      socket  4.000000e+00 2024-01-16 10:07:00+00:00
58  recommendationservice         mem  4.599915e+07 2024-01-16 10:07:00+00:00
57  recommendationservice  latency-90  9.716867e-03 2024-01-16 10:07:00+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse variables from kernel: result_df (contains p95 and exceed_count_in_window), df_window (incident-window rows)
# If they are not present, try to recreate minimally (but they should exist per prior steps)
try:
    result_df
    df_window
except NameError:
    # Minimal recreation if needed (shouldn't happen in this session)
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    components = [
        "checkoutservice","currencyservice","emailservice","productcatalogservice","recommendationservice"
    ]
    df_m_sub = df_metric[df_metric['cmdb_id'].isin(components)].copy()
    # global p95 summary
    result_df = df_m_sub.groupby(['cmdb_id','kpi_name']).agg(
        count=('value','size'),
        min=('value','min'),
        max=('value','max'),
        p50=('value', lambda x: np.percentile(x.dropna(),50) if len(x.dropna())>0 else np.nan),
        p90=('value', lambda x: np.percentile(x.dropna(),90) if len(x.dropna())>0 else np.nan),
        p95=('value', lambda x: np.percentile(x.dropna(),95) if len(x.dropna())>0 else np.nan),
    ).reset_index()
    window_start = pd.to_datetime("2024-01-16 10:04:37", utc=True)
    window_end   = pd.to_datetime("2024-01-16 10:34:37", utc=True)
    df_window = df_m_sub[(df_m_sub['ts']>=window_start) & (df_m_sub['ts']<=window_end)].copy()
    # compute counts/exceeds as before
    df_win_with_p95 = df_window.merge(result_df[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
    df_win_with_p95['exceed'] = df_win_with_p95['value'] >= df_win_with_p95['p95']
    exceed_counts = df_win_with_p95.groupby(['cmdb_id','kpi_name']).agg(exceed_count_in_window=('exceed','sum')).reset_index()
    result_df = result_df.merge(
        df_window.groupby(['cmdb_id','kpi_name']).agg(count_in_window=('value','size')).reset_index(),
        on=['cmdb_id','kpi_name'], how='left'
    ).merge(exceed_counts, on=['cmdb_id','kpi_name'], how='left').fillna({'count_in_window':0,'exceed_count_in_window':0})
    result_df['count_in_window']=result_df['count_in_window'].astype(int)
    result_df['exceed_count_in_window']=result_df['exceed_count_in_window'].astype(int)

# Identify groups with at least one exceed in window
exceed_groups = result_df[result_df['exceed_count_in_window'] > 0][['cmdb_id','kpi_name','p95','exceed_count_in_window']]
exceed_groups = exceed_groups.reset_index(drop=True)

runs = []  # accumulate run dicts

# For each group, find consecutive runs of rows in df_window where value >= p95
for _, row in exceed_groups.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = float(row['p95'])
    # subset incident-window rows for this group, sorted by ts
    dfg = df_window[(df_window['cmdb_id']==cmdb) & (df_window['kpi_name']==kpi)].sort_values('ts').copy()
    if dfg.empty:
        continue
    # mark exceed boolean
    dfg['exceed'] = dfg['value'] >= p95
    # take only exceed rows (runs defined by consecutive exceed timestamps)
    dfg_ex = dfg[dfg['exceed']].copy()
    if dfg_ex.empty:
        continue
    # compute time gap between consecutive exceed rows
    dfg_ex['ts_diff'] = dfg_ex['ts'].diff().fillna(pd.Timedelta(seconds=0))
    # new run when ts_diff > 1 minute
    dfg_ex['new_run'] = (dfg_ex['ts_diff'] > pd.Timedelta(minutes=1)).astype(int)
    # run id as cumulative sum of new_run
    dfg_ex['run_id'] = dfg_ex['new_run'].cumsum()
    # aggregate per run
    agg = dfg_ex.groupby('run_id').agg(
        cmdb_id=('cmdb_id','first'),
        kpi_name=('kpi_name','first'),
        run_start_ts=('ts','min'),
        run_end_ts=('ts','max'),
        run_length=('ts','size'),
        run_max_value=('value','max')
    ).reset_index(drop=True)
    # add global p95 and severity_ratio
    agg['global_p95'] = p95
    # avoid division by zero
    agg['severity_ratio'] = np.where(agg['global_p95']==0, np.nan, agg['run_max_value'] / agg['global_p95'])
    # format timestamps as ISO UTC with +00:00
    agg['run_start_ts'] = agg['run_start_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00')
    agg['run_end_ts']   = agg['run_end_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00')
    # collect
    runs.append(agg[['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','run_max_value','global_p95','severity_ratio']])

# Concatenate runs
if runs:
    runs_df = pd.concat(runs, ignore_index=True)
else:
    runs_df = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','run_max_value','global_p95','severity_ratio'])

# Sort as requested: run_length desc, then severity_ratio desc
runs_df_sorted = runs_df.sort_values(by=['run_length','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

# Limit to top 20 runs
top_runs = runs_df_sorted.head(20).copy()

# Brief summary: which component-KPI had the longest run_length and which had highest severity_ratio
if not top_runs.empty:
    longest = top_runs.loc[top_runs['run_length'].idxmax()]
    highest_sev = top_runs.loc[top_runs['severity_ratio'].idxmax()]
    summary_line = (
        f"Longest run: {longest['cmdb_id']} | {longest['kpi_name']} | length={int(longest['run_length'])} minutes | "
        f"start={longest['run_start_ts']} end={longest['run_end_ts']}; "
        f"Highest severity: {highest_sev['cmdb_id']} | {highest_sev['kpi_name']} | severity_ratio={float(highest_sev['severity_ratio']):.3f} "
        f"(run_max={float(highest_sev['run_max_value']):.6g} / p95={float(highest_sev['global_p95']):.6g})"
    )
else:
    summary_line = "No exceedance runs found in the incident window."

# Keep output compact: top_runs DataFrame and summary_line
top_runs, summary_line

```
Out[3]:
```
Summary of detected exceedance runs (top 20, sorted by length then severity):

- Total runs returned (top-20): 20.
- Longest run: currencyservice | socket — 25 minutes (2024-01-16T10:07:00+00:00 to 2024-01-16T10:31:00+00:00).
- Highest severity ratio: checkoutservice | cpu — severity_ratio ≈ 1.239 (run_max = 0.533468, global_p95 = 0.430583).

Top runs (first 6, compact):
1. currencyservice | socket — run_length 25 min, run_max 3.633333, p95 3.000000, severity 1.211  
2. checkoutservice | socket — run_length 25 min, run_max 9.000000, p95 9.000000, severity 1.000  
3. emailservice | socket — run_length 25 min, run_max 3.000000, p95 3.000000, severity 1.000  
4. recommendationservice | socket — run_length 25 min, run_max 4.000000, p95 4.000000, severity 1.000  
5. currencyservice | error — run_length 25 min, run_max 0.000000, p95 0.000000, severity N/A (p95 = 0)  
6. emailservice | diskio — run_length 25 min, run_max 0.000000, p95 0.000000, severity N/A (p95 = 0)

Additional notable runs (shorter lengths with elevated severity):
- productcatalogservice | socket — 15 min, severity 1.000 (run_max 4, p95 4).  
- Several 2-minute runs for workload/cpu/latency KPIs with severity ratios between ~1.03 and ~1.16.
- Single-minute spikes for some checkoutservice and emailservice CPU/latency KPIs with severity > 1.

Brief interpretation:
- The longest and most persistent exceedances are for the "socket" KPI across multiple services (many lasted the full incident window).  
- The single highest relative spike (severity_ratio) was a short CPU spike for checkoutservice.  
- Some KPIs have p95 = 0 (error/diskio), producing runs with run_max = 0 (severity undefined) — these should be interpreted carefully.

If you want, I can:
- Provide the full top-20 run table as CSV-like rows, or  
- Show all incident-window rows for a specific run (to inspect exact minute-by-minute values).

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name               run_start_ts                 run_end_ts  run_length  run_max_value  global_p95  severity_ratio
0         currencyservice      socket  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       3.633333    3.000000        1.211111
1         checkoutservice      socket  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       9.000000    9.000000        1.000000
2            emailservice      socket  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       3.000000    3.000000        1.000000
3   recommendationservice      socket  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       4.000000    4.000000        1.000000
4         currencyservice       error  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       0.000000    0.000000             NaN
5            emailservice      diskio  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       0.000000    0.000000             NaN
6   recommendationservice      diskio  2024-01-16T10:07:00+00:00  2024-01-16T10:31:00+00:00          25       0.000000    0.000000             NaN
7   productcatalogservice      socket  2024-01-16T10:07:00+00:00  2024-01-16T10:21:00+00:00          15       4.000000    4.000000        1.000000
8         checkoutservice    workload  2024-01-16T10:19:00+00:00  2024-01-16T10:20:00+00:00           2       1.280150    1.102337        1.161306
9            emailservice    workload  2024-01-16T10:19:00+00:00  2024-01-16T10:20:00+00:00           2       1.259867    1.105677        1.139453
10  productcatalogservice    workload  2024-01-16T10:12:00+00:00  2024-01-16T10:13:00+00:00           2     153.537783  145.152113        1.057772
11        currencyservice    workload  2024-01-16T10:11:00+00:00  2024-01-16T10:12:00+00:00           2      85.664367   82.837807        1.034122
12  recommendationservice         cpu  2024-01-16T10:29:00+00:00  2024-01-16T10:30:00+00:00           2       3.115075    3.013377        1.033749
13        currencyservice         cpu  2024-01-16T10:11:00+00:00  2024-01-16T10:12:00+00:00           2      30.692099   29.719117        1.032739
14  productcatalogservice  latency-50  2024-01-16T10:30:00+00:00  2024-01-16T10:31:00+00:00           2       0.002076    0.002035        1.020169
15        currencyservice  latency-90  2024-01-16T10:12:00+00:00  2024-01-16T10:13:00+00:00           2       0.067436    0.066264        1.017681
16        checkoutservice         cpu  2024-01-16T10:20:00+00:00  2024-01-16T10:20:00+00:00           1       0.533468    0.430583        1.238943
17        checkoutservice  latency-50  2024-01-16T10:14:00+00:00  2024-01-16T10:14:00+00:00           1       0.147779    0.138724        1.065273
18  recommendationservice    workload  2024-01-16T10:13:00+00:00  2024-01-16T10:13:00+00:00           1      20.175683   19.038737        1.059718
19           emailservice         cpu  2024-01-16T10:20:00+00:00  2024-01-16T10:20:00+00:00           1       0.332708    0.324668        1.024761, 'Longest run: currencyservice | socket | length=25 minutes | start=2024-01-16T10:07:00+00:00 end=2024-01-16T10:31:00+00:00; Highest severity: checkoutservice | cpu | severity_ratio=1.239 (run_max=0.533468 / p95=0.430583)')```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is timezone-aware UTC
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    df_trace['ts'] = pd.to_datetime(df_trace['ts'], utc=True)

# Services to match anywhere in trace_name
services = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# Build regex to match any service name
pat = "|".join(services)

# Filter trace rows whose trace_name contains any of the service names
mask = df_trace['trace_name'].astype(str).str.contains(pat, regex=True)
df_tr_sub = df_trace[mask].copy()

# Compute global thresholds (using entire available series for each (cmdb_id, trace_name))
group_keys = ['cmdb_id', 'trace_name']
global_stats = (
    df_tr_sub.groupby(group_keys)['value']
    .agg([
        ('count', 'size'),
        ('p50', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan),
        ('p90', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna())>0 else np.nan),
        ('p95', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan),
    ])
    .reset_index()
)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-16 10:04:37", utc=True)
window_end   = pd.to_datetime("2024-01-16 10:34:37", utc=True)

# Filter to incident-window subset
df_tr_window = df_tr_sub[(df_tr_sub['ts'] >= window_start) & (df_tr_sub['ts'] <= window_end)].copy()

# Compute count_in_window per group
count_in_window = (
    df_tr_window.groupby(group_keys)
    .agg(count_in_window=('value','size'))
    .reset_index()
)

# Merge global p95 into window rows to evaluate exceedances
if not df_tr_window.empty:
    df_tw = df_tr_window.merge(global_stats[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')
    df_tw['exceed'] = df_tw['value'] >= df_tw['p95']
    exceed_counts = (
        df_tw.groupby(group_keys)
        .agg(exceed_count_in_window=('exceed','sum'))
        .reset_index()
    )
    earliest_exceed = (
        df_tw[df_tw['exceed']]
        .groupby(group_keys)
        .agg(earliest_exceed_ts=('ts','min'))
        .reset_index()
    )
    if not earliest_exceed.empty:
        earliest_exceed['earliest_exceed_ts'] = earliest_exceed['earliest_exceed_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00')
else:
    exceed_counts = pd.DataFrame(columns=group_keys + ['exceed_count_in_window'])
    earliest_exceed = pd.DataFrame(columns=group_keys + ['earliest_exceed_ts'])

# Build summary table by merging global and window metrics
trace_summary = global_stats.merge(count_in_window, on=group_keys, how='left')
trace_summary = trace_summary.merge(exceed_counts, on=group_keys, how='left')
trace_summary = trace_summary.merge(earliest_exceed, on=group_keys, how='left')

# Fill NaNs for groups with no window rows
trace_summary['count_in_window'] = trace_summary['count_in_window'].fillna(0).astype(int)
trace_summary['exceed_count_in_window'] = trace_summary['exceed_count_in_window'].fillna(0).astype(int)
trace_summary['earliest_exceed_ts'] = trace_summary['earliest_exceed_ts'].where(trace_summary['earliest_exceed_ts'].notna(), None)

# Sort by exceed_count_in_window descending then count_in_window desc, limit top 20
trace_summary_top = trace_summary.sort_values(['exceed_count_in_window','count_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Sample up to 5 trace rows from the incident-window subset for inspection
trace_samples = df_tr_window.sort_values('ts').loc[:, ['cmdb_id','trace_name','value','ts']].head(5)

# Keep output compact: selected columns only
trace_summary_top = trace_summary_top[['cmdb_id','trace_name','count','p50','p90','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts']]

trace_summary_top, trace_samples

```
Out[4]:
```
Summary of trace analysis for the five services in the incident window (2024-01-16T10:04:37 to 2024-01-16T10:34:37 UTC):

- Rows returned (top 20 sorted by exceed_count_in_window): 20 groups (mostly trace.error_rate entries across services and a few other related entries).
- Key finding: nearly all top groups are trace.error_rate metrics (e.g., trace.from_checkoutservice.error_rate, trace.to_checkoutservice.error_rate, trace.to_currencyservice.error_rate, etc.). For each of these groups:
  - Global p95 = 0.0.
  - count (full series) = 25.
  - count_in_window = 25 and exceed_count_in_window = 25.
  - earliest_exceed_ts = 2024-01-16T10:07:00+00:00.
- Interpretation note: because the global P95 for these error_rate traces is 0.0, every value >= 0 is counted as an "exceedance". This makes those exceedance counts uninformative — the p95 = 0 condition likely reflects very sparse/nonpositive error_rate values or the metric normalization. Treat these "exceeds" with caution.
- Example non-error trace samples from the window (up to 5 rows):
  - checkoutservice | trace.from_checkoutservice.duration_mean = 0.003768 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.row_count = 207.0 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.error_rate = 0.0 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.duration_p95 = 0.000030 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.duration_mean = 0.000016 at 2024-01-16T10:07:00+00:00

Recommended next steps:
- Ignore or re-evaluate groups with p95 = 0 when looking for meaningful exceedances (choose p90 or another nonzero threshold, or focus on metrics with nonzero p95).  
- Inspect trace metrics with nonzero p95 (duration, row_count, workload, latency) for meaningful spikes during the incident window.  
- If you want, I can re-run the table excluding p95==0 groups or produce a list of trace groups with nonzero p95 and their exceedance summaries.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  count  p50  p90  p95  count_in_window  exceed_count_in_window         earliest_exceed_ts
0         checkoutservice        trace.from_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
1         checkoutservice          trace.to_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
2         checkoutservice          trace.to_currencyservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
3         checkoutservice             trace.to_emailservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
4         checkoutservice    trace.to_productcatalogservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
5         currencyservice        trace.from_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
6            emailservice        trace.from_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
7         frontendservice          trace.to_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
8         frontendservice          trace.to_currencyservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
9         frontendservice    trace.to_productcatalogservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
10        frontendservice    trace.to_recommendationservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
11         paymentservice        trace.from_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
12  productcatalogservice        trace.from_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
13  productcatalogservice  trace.from_recommendationservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
14  recommendationservice  trace.from_recommendationservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
15  recommendationservice    trace.to_productcatalogservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
16  recommendationservice    trace.to_recommendationservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
17                   root          trace.to_checkoutservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
18                   root          trace.to_currencyservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00
19                   root             trace.to_emailservice.error_rate     25  0.0  0.0  0.0               25                      25  2024-01-16T10:07:00+00:00,                    cmdb_id                                    trace_name       value                        ts
0          checkoutservice      trace.from_checkoutservice.duration_mean    0.003768 2024-01-16 10:07:00+00:00
119  recommendationservice      trace.to_productcatalogservice.row_count  207.000000 2024-01-16 10:07:00+00:00
118  recommendationservice     trace.to_productcatalogservice.error_rate    0.000000 2024-01-16 10:07:00+00:00
117  recommendationservice   trace.to_productcatalogservice.duration_p95    0.000030 2024-01-16 10:07:00+00:00
116  recommendationservice  trace.to_productcatalogservice.duration_mean    0.000016 2024-01-16 10:07:00+00:00)```
```

In [ ]:
# In[5]:

import numpy as np
import pandas as pd

# Reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure ts exists and is UTC-aware
if 'ts' not in df_trace.columns:
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    df_trace['ts'] = pd.to_datetime(df_trace['ts'], utc=True)

# Services to match
services = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]
pat = "|".join(services)

# Filter trace rows whose trace_name contains any of the service names
df_tr_sub = df_trace[df_trace['trace_name'].astype(str).str.contains(pat, regex=True)].copy()

# Compute global stats (using full series) per (cmdb_id, trace_name)
group_keys = ['cmdb_id', 'trace_name']
global_stats = (
    df_tr_sub.groupby(group_keys)['value']
    .agg([
        ('count', 'size'),
        ('p50', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan),
        ('p90', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna())>0 else np.nan),
        ('p95', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan),
    ])
    .reset_index()
)

# Keep only groups with global p95 > 0
global_pos = global_stats[global_stats['p95'] > 0].copy()

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-16 10:04:37", utc=True)
window_end   = pd.to_datetime("2024-01-16 10:34:37", utc=True)

# Incident-window subset
df_tr_window = df_tr_sub[(df_tr_sub['ts'] >= window_start) & (df_tr_sub['ts'] <= window_end)].copy()

# count_in_window per group
count_in_window = df_tr_window.groupby(group_keys).agg(count_in_window=('value','size')).reset_index()

# Merge global p95 into window rows to evaluate exceedances (only for groups with p95>0)
if not df_tr_window.empty and not global_pos.empty:
    df_tw = df_tr_window.merge(global_pos[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='inner')
    # mark exceed
    df_tw['exceed'] = df_tw['value'] >= df_tw['p95']
    exceed_counts = df_tw.groupby(group_keys).agg(exceed_count_in_window=('exceed','sum')).reset_index()
    earliest_exceed = (
        df_tw[df_tw['exceed']]
        .groupby(group_keys)
        .agg(earliest_exceed_ts=('ts','min'))
        .reset_index()
    )
    if not earliest_exceed.empty:
        earliest_exceed['earliest_exceed_ts'] = earliest_exceed['earliest_exceed_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00')
else:
    exceed_counts = pd.DataFrame(columns=group_keys + ['exceed_count_in_window'])
    earliest_exceed = pd.DataFrame(columns=group_keys + ['earliest_exceed_ts'])

# Merge results into final summary for groups with p95>0
trace_summary = global_pos.merge(count_in_window, on=group_keys, how='left')
trace_summary = trace_summary.merge(exceed_counts, on=group_keys, how='left')
trace_summary = trace_summary.merge(earliest_exceed, on=group_keys, how='left')

# Fill missing counts with 0 and missing earliest_exceed_ts with None
trace_summary['count_in_window'] = trace_summary['count_in_window'].fillna(0).astype(int)
trace_summary['exceed_count_in_window'] = trace_summary['exceed_count_in_window'].fillna(0).astype(int)
trace_summary['earliest_exceed_ts'] = trace_summary['earliest_exceed_ts'].where(trace_summary['earliest_exceed_ts'].notna(), None)

# Sort by exceed_count_in_window desc then p95 desc, limit top 20
trace_summary_top = trace_summary.sort_values(['exceed_count_in_window','p95'], ascending=[False, False]).head(20).reset_index(drop=True)

# Select compact columns
trace_summary_top = trace_summary_top[['cmdb_id','trace_name','count','p50','p90','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts']]

# Up to 5 sample trace rows from the incident-window subset
trace_samples = df_tr_window.sort_values('ts').loc[:, ['cmdb_id','trace_name','value','ts']].head(5)

# Display compact results
trace_summary_top, trace_samples

```
Out[5]:
```
Summary of trace findings for the five services during 2024-01-16T10:04:37 → 2024-01-16T10:34:37 UTC:

- Returned groups (filtered to trace_name mentioning the services and with global p95 > 0): 20 top groups shown.
- Biggest signal: several "root → service" row_count traces have many exceedances:
  - root → to_emailservice.row_count, root → to_productcatalogservice.row_count, root → to_recommendationservice.row_count all have global p95 = 24 and show 23 exceedances out of 25 window points. Earliest exceed timestamp for those is 2024-01-16T10:08:00+00:00.
  - root → to_currencyservice.row_count has p95 = 12 and 23 exceedances, earliest exceed 2024-01-16T10:08:00+00:00.
  - Interpretation: a substantial portion of row_count samples in the window exceed the normal 95th percentile for these root->service row_count traces, with a surge visible starting ~10:08 UTC.
- Smaller but notable exceedances (count_in_window = 25 but exceed_count_in_window = 2):
  - frontendservice → to_productcatalogservice.row_count (p95 ≈ 3834.6), earliest exceed 2024-01-16T10:08:00+00:00.
  - frontendservice → to_currencyservice.row_count (p95 ≈ 2392.0), earliest exceed 2024-01-16T10:11:00+00:00.
  - frontendservice → to_recommendationservice.row_count and several productcatalog/recommendation/checkout groups (row_count) show 2 exceedances, usually with earliest exceed ~10:08 or ~10:19.
- Other groups with nonzero p95 and 2 exceedances include checkoutservice/productcatalogservice/recommendationservice row_count traces (earliest exceed ~10:19 UTC).
- Example sample rows from the incident window (for quick inspection):
  - checkoutservice | trace.from_checkoutservice.duration_mean = 0.003768 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.row_count = 207.0 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.error_rate = 0.0 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.duration_p95 = 0.000030 at 2024-01-16T10:07:00+00:00
  - recommendationservice | trace.to_productcatalogservice.duration_mean = 0.000016 at 2024-01-16T10:07:00+00:00

Concise interpretation / next steps:
- There is a clear surge in row_count metrics observed from the "root" traces beginning around 10:08 UTC impacting multiple downstream services (email, productcatalog, recommendation, currency). This looks like increased request/row traffic rather than isolated latency spikes.
- Smaller clusters of exceedances occur ~10:19 UTC for checkoutservice-related traces and for several frontend→service row_count traces (2-minute spikes).
- Recommended follow-ups: inspect request/trace details and service logs around ~10:08 and ~10:19 UTC for the implicated services to determine whether traffic spikes, routing changes, or upstream behavior caused the row_count increases. If you want, I can (a) show the full top-20 table rows, (b) display all incident-window trace rows for a chosen group, or (c) correlate these times with metric anomalies already detected.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                  trace_name  count     p50     p90     p95  count_in_window  exceed_count_in_window         earliest_exceed_ts
0                    root             trace.to_emailservice.row_count     25    24.0    24.0    24.0               25                      23  2024-01-16T10:08:00+00:00
1                    root    trace.to_productcatalogservice.row_count     25    24.0    24.0    24.0               25                      23  2024-01-16T10:08:00+00:00
2                    root    trace.to_recommendationservice.row_count     25    24.0    24.0    24.0               25                      23  2024-01-16T10:08:00+00:00
3                    root          trace.to_currencyservice.row_count     25    12.0    12.0    12.0               25                      23  2024-01-16T10:08:00+00:00
4         frontendservice    trace.to_productcatalogservice.row_count     25  3484.0  3767.4  3834.6               25                       2  2024-01-16T10:08:00+00:00
5         frontendservice          trace.to_currencyservice.row_count     25  2212.0  2325.6  2392.0               25                       2  2024-01-16T10:11:00+00:00
6         frontendservice    trace.to_recommendationservice.row_count     25   535.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00
7   productcatalogservice  trace.from_recommendationservice.row_count     25   537.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00
8   recommendationservice  trace.from_recommendationservice.row_count     25   535.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00
9   recommendationservice    trace.to_productcatalogservice.row_count     25   537.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00
10  recommendationservice    trace.to_recommendationservice.row_count     25   535.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00
11        checkoutservice        trace.from_checkoutservice.row_count     25   211.0   284.6   306.6               25                       2  2024-01-16T10:19:00+00:00
12        checkoutservice          trace.to_checkoutservice.row_count     25   211.0   284.6   306.6               25                       2  2024-01-16T10:19:00+00:00
13                   root          trace.to_checkoutservice.row_count     25   111.0   141.2   151.6               25                       2  2024-01-16T10:19:00+00:00
14        checkoutservice          trace.to_currencyservice.row_count     25    87.0   117.2   127.6               25                       2  2024-01-16T10:19:00+00:00
15        currencyservice        trace.from_checkoutservice.row_count     25    87.0   117.2   127.6               25                       2  2024-01-16T10:19:00+00:00
16        checkoutservice    trace.to_productcatalogservice.row_count     25    63.0    83.8    87.8               25                       2  2024-01-16T10:19:00+00:00
17  productcatalogservice        trace.from_checkoutservice.row_count     25    63.0    83.8    87.8               25                       2  2024-01-16T10:19:00+00:00
18        checkoutservice             trace.to_emailservice.row_count     25    26.0    33.4    35.8               25                       2  2024-01-16T10:19:00+00:00
19           emailservice        trace.from_checkoutservice.row_count     25    26.0    33.4    35.8               25                       2  2024-01-16T10:19:00+00:00,                    cmdb_id                                    trace_name       value                        ts
0          checkoutservice      trace.from_checkoutservice.duration_mean    0.003768 2024-01-16 10:07:00+00:00
119  recommendationservice      trace.to_productcatalogservice.row_count  207.000000 2024-01-16 10:07:00+00:00
118  recommendationservice     trace.to_productcatalogservice.error_rate    0.000000 2024-01-16 10:07:00+00:00
117  recommendationservice   trace.to_productcatalogservice.duration_p95    0.000030 2024-01-16 10:07:00+00:00
116  recommendationservice  trace.to_productcatalogservice.duration_mean    0.000016 2024-01-16 10:07:00+00:00)```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd

# Reuse df_log if present; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is timezone-aware UTC
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    df_log['ts'] = pd.to_datetime(df_log['ts'], utc=True)

# Components to restrict
components = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# Restrict to those cmdb_id values
df_log_sub = df_log[df_log['cmdb_id'].isin(components)].copy()

# Compute global thresholds (full series) per (cmdb_id, log_name)
group_keys = ['cmdb_id', 'log_name']
global_stats = (
    df_log_sub.groupby(group_keys)['value']
    .agg([
        ('count', 'size'),
        ('p50', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna())>0 else np.nan),
        ('p90', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna())>0 else np.nan),
        ('p95', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna())>0 else np.nan),
    ])
    .reset_index()
)

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-16 10:04:37", utc=True)
window_end   = pd.to_datetime("2024-01-16 10:34:37", utc=True)

# Filter to incident-window subset
df_log_window = df_log_sub[(df_log_sub['ts'] >= window_start) & (df_log_sub['ts'] <= window_end)].copy()

# count_in_window per group
count_in_window = df_log_window.groupby(group_keys).agg(count_in_window=('value','size')).reset_index()

# Merge global p95 into window rows to evaluate exceedances
if not df_log_window.empty and not global_stats.empty:
    df_lw = df_log_window.merge(global_stats[['cmdb_id','log_name','p95']], on=['cmdb_id','log_name'], how='left')
    df_lw['exceed'] = df_lw['value'] >= df_lw['p95']
    exceed_counts = df_lw.groupby(group_keys).agg(exceed_count_in_window=('exceed','sum')).reset_index()
    earliest_exceed = (
        df_lw[df_lw['exceed']]
        .groupby(group_keys)
        .agg(earliest_exceed_ts=('ts','min'))
        .reset_index()
    )
    if not earliest_exceed.empty:
        earliest_exceed['earliest_exceed_ts'] = earliest_exceed['earliest_exceed_ts'].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000','+00:00')
else:
    exceed_counts = pd.DataFrame(columns=group_keys + ['exceed_count_in_window'])
    earliest_exceed = pd.DataFrame(columns=group_keys + ['earliest_exceed_ts'])

# Build final summary table by merging global and window metrics
log_summary = global_stats.merge(count_in_window, on=group_keys, how='left')
log_summary = log_summary.merge(exceed_counts, on=group_keys, how='left')
log_summary = log_summary.merge(earliest_exceed, on=group_keys, how='left')

# Fill missing counts with 0 and earliest_exceed_ts with None
log_summary['count_in_window'] = log_summary['count_in_window'].fillna(0).astype(int)
log_summary['exceed_count_in_window'] = log_summary['exceed_count_in_window'].fillna(0).astype(int)
log_summary['earliest_exceed_ts'] = log_summary['earliest_exceed_ts'].where(log_summary['earliest_exceed_ts'].notna(), None)

# Select and order requested columns; show only existing groups (should be small)
results_df = log_summary[['cmdb_id','log_name','count','p50','p90','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts']].reset_index(drop=True)

# Up to 5 sample log rows from the incident-window subset for inspection
sample_rows = df_log_window.sort_values('ts').loc[:, ['cmdb_id','log_name','value','ts']].head(5)

# Compact display of results
results_df, sample_rows

```
Out[6]:
```
Summary of log analysis for the five components (incident window 2024-01-16T10:04:37 → 2024-01-16T10:34:37 UTC):

Groups found (compact):

- checkoutservice | log.error_count — count=25, p95=0.0, count_in_window=25, exceed_count_in_window=25, earliest_exceed_ts=2024-01-16T10:07:00+00:00  
- checkoutservice | log.row_count   — count=25, p95=107.4, count_in_window=25, exceed_count_in_window=2,  earliest_exceed_ts=2024-01-16T10:19:00+00:00  
- currencyservice | log.error_count — count=25, p95=0.0, count_in_window=25, exceed_count_in_window=25, earliest_exceed_ts=2024-01-16T10:07:00+00:00  
- currencyservice | log.row_count   — count=25, p95=2486.2, count_in_window=25, exceed_count_in_window=2, earliest_exceed_ts=2024-01-16T10:11:00+00:00  
- emailservice | log.error_count    — count=25, p95=0.0, count_in_window=25, exceed_count_in_window=25, earliest_exceed_ts=2024-01-16T10:07:00+00:00  
- emailservice | log.row_count      — count=25, p95=35.8, count_in_window=25, exceed_count_in_window=2,  earliest_exceed_ts=2024-01-16T10:19:00+00:00  
- recommendationservice | log.error_count — count=25, p95=0.0, count_in_window=25, exceed_count_in_window=25, earliest_exceed_ts=2024-01-16T10:07:00+00:00  
- recommendationservice | log.row_count   — count=25, p95=588.2, count_in_window=25, exceed_count_in_window=2,  earliest_exceed_ts=2024-01-16T10:08:00+00:00

Sample log rows from the incident window (up to 5):
- checkoutservice | log.error_count = 0 at 2024-01-16T10:07:00+00:00  
- checkoutservice | log.row_count   = 21 at 2024-01-16T10:07:00+00:00  
- currencyservice | log.error_count = 0 at 2024-01-16T10:07:00+00:00  
- currencyservice | log.row_count   = 935 at 2024-01-16T10:07:00+00:00  
- emailservice | log.error_count    = 0 at 2024-01-16T10:07:00+00:00

Interpretation / notes:
- All log.error_count groups have global p95 = 0.0, so every value >= 0 is counted as an "exceedance" (exceed_count_in_window = 25). This is uninformative: p95==0 should be treated cautiously (consider using p90 or a nonzero threshold or inspect raw log entries).
- log.row_count groups have meaningful nonzero p95 values and show a small number of exceedances (2 each) at specific times (notably around 10:08, 10:11, and 10:19 UTC depending on service). These are the actionable signals to inspect further.
- Recommended next steps: focus on the log.row_count exceedance timestamps per service (inspect logs and traces around the earliest_exceed_ts values) and disregard or re-threshold log.error_count groups unless you confirm nonzero error events.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id         log_name  count     p50     p90     p95  count_in_window  exceed_count_in_window         earliest_exceed_ts
0        checkoutservice  log.error_count     25     0.0     0.0     0.0               25                      25  2024-01-16T10:07:00+00:00
1        checkoutservice    log.row_count     25    78.0   100.6   107.4               25                       2  2024-01-16T10:19:00+00:00
2        currencyservice  log.error_count     25     0.0     0.0     0.0               25                      25  2024-01-16T10:07:00+00:00
3        currencyservice    log.row_count     25  2286.0  2430.6  2486.2               25                       2  2024-01-16T10:11:00+00:00
4           emailservice  log.error_count     25     0.0     0.0     0.0               25                      25  2024-01-16T10:07:00+00:00
5           emailservice    log.row_count     25    26.0    33.4    35.8               25                       2  2024-01-16T10:19:00+00:00
6  recommendationservice  log.error_count     25     0.0     0.0     0.0               25                      25  2024-01-16T10:07:00+00:00
7  recommendationservice    log.row_count     25   537.0   577.8   588.2               25                       2  2024-01-16T10:08:00+00:00,            cmdb_id         log_name  value                        ts
4  checkoutservice  log.error_count      0 2024-01-16 10:07:00+00:00
5  checkoutservice    log.row_count     21 2024-01-16 10:07:00+00:00
6  currencyservice  log.error_count      0 2024-01-16 10:07:00+00:00
7  currencyservice    log.row_count    935 2024-01-16 10:07:00+00:00
8     emailservice  log.error_count      0 2024-01-16 10:07:00+00:00)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# candidate services
services = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice",
]

# Reuse runs_df if present (contains all metric runs). If not present, rebuild from earlier state.
try:
    runs_df
except NameError:
    # Rebuild runs_df from df_window and result_df (should exist from prior steps)
    # This re-creates the runs (only for groups with exceedances) following earlier logic.
    runs = []
    for _, row in result_df[result_df['exceed_count_in_window']>0].iterrows():
        cmdb = row['cmdb_id']; kpi = row['kpi_name']; p95 = float(row['p95'])
        if p95 <= 0:
            continue
        dfg = df_window[(df_window['cmdb_id']==cmdb) & (df_window['kpi_name']==kpi)].sort_values('ts').copy()
        if dfg.empty:
            continue
        dfg['exceed'] = dfg['value'] >= p95
        dfg_ex = dfg[dfg['exceed']].copy()
        if dfg_ex.empty:
            continue
        dfg_ex['ts_diff'] = dfg_ex['ts'].diff().fillna(pd.Timedelta(seconds=0))
        dfg_ex['new_run'] = (dfg_ex['ts_diff'] > pd.Timedelta(minutes=1)).astype(int)
        dfg_ex['run_id'] = dfg_ex['new_run'].cumsum()
        agg = dfg_ex.groupby('run_id').agg(
            cmdb_id=('cmdb_id','first'),
            kpi_name=('kpi_name','first'),
            run_start_ts=('ts','min'),
            run_end_ts=('ts','max'),
            run_length=('ts','size'),
            run_max_value=('value','max')
        ).reset_index(drop=True)
        agg['global_p95'] = p95
        agg['severity_ratio'] = np.where(agg['global_p95']==0, np.nan, agg['run_max_value']/agg['global_p95'])
        # keep datetimes as tz-aware; format later if needed
        runs.append(agg[['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','run_max_value','global_p95','severity_ratio']])
    runs_df = pd.concat(runs, ignore_index=True) if runs else pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length','run_max_value','global_p95','severity_ratio'])

# Normalize runs_df timestamps to datetime (tz-aware)
if not runs_df.empty:
    runs_df['run_start_ts'] = pd.to_datetime(runs_df['run_start_ts'], utc=True)
    runs_df['run_end_ts'] = pd.to_datetime(runs_df['run_end_ts'], utc=True)

# Reuse trace_summary_top (groups with p95>0) if present; otherwise use trace_summary
trace_groups = None
if 'trace_summary_top' in globals():
    trace_groups = trace_summary_top.copy()
elif 'trace_summary' in globals():
    trace_groups = trace_summary[trace_summary['p95']>0].copy()
else:
    # fallback: compute minimal trace_summary from df_tr_window/global_stats if available
    trace_groups = pd.DataFrame(columns=['cmdb_id','trace_name','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts'])

# Ensure earliest_exceed_ts are parsed to datetimes where present
if not trace_groups.empty:
    trace_groups['earliest_exceed_dt'] = pd.to_datetime(trace_groups['earliest_exceed_ts'], utc=True)

# Reuse log results (results_df) if present
if 'results_df' in globals():
    log_groups = results_df.copy()
else:
    log_groups = pd.DataFrame(columns=['cmdb_id','log_name','p95','count_in_window','exceed_count_in_window','earliest_exceed_ts'])

# Parse earliest_exceed_ts for logs
if not log_groups.empty:
    log_groups['earliest_exceed_dt'] = pd.to_datetime(log_groups['earliest_exceed_ts'], utc=True)

# Build one-row-per-service summary
rows = []
for svc in services:
    # Metric: runs with global_p95>0 and severity_ratio>1.05
    metric_candidate = pd.DataFrame()
    if not runs_df.empty:
        metric_candidate = runs_df[
            (runs_df['cmdb_id']==svc) &
            (runs_df['global_p95']>0) &
            (runs_df['severity_ratio']>1.05)
        ].copy()
    if metric_candidate.empty:
        metric_fields = {
            'earliest_metric_anom_ts': None,
            'metric_kpi_name': None,
            'metric_run_length': None,
            'metric_run_max': None,
            'metric_global_p95': None,
            'metric_severity_ratio': None
        }
    else:
        # pick earliest run_start_ts
        metric_candidate = metric_candidate.sort_values('run_start_ts')
        m = metric_candidate.iloc[0]
        metric_fields = {
            'earliest_metric_anom_ts': m['run_start_ts'].tz_convert('UTC'),
            'metric_kpi_name': m['kpi_name'],
            'metric_run_length': int(m['run_length']),
            'metric_run_max': float(m['run_max_value']),
            'metric_global_p95': float(m['global_p95']),
            'metric_severity_ratio': float(m['severity_ratio'])
        }

    # Trace: earliest trace exceed ts for groups with p95>0 and exceed_count_in_window>0
    trace_candidate = pd.DataFrame()
    if (trace_groups is not None) and (not trace_groups.empty):
        trace_candidate = trace_groups[
            (trace_groups['cmdb_id']==svc) &
            (trace_groups['p95'] > 0) &
            (trace_groups['exceed_count_in_window'] > 0)
        ].copy()
    if trace_candidate.empty:
        trace_fields = {'earliest_trace_exceed_ts': None, 'trace_name': None}
    else:
        trace_candidate = trace_candidate.sort_values('earliest_exceed_dt')
        t = trace_candidate.iloc[0]
        trace_fields = {'earliest_trace_exceed_ts': pd.to_datetime(t['earliest_exceed_ts'], utc=True), 'trace_name': t['trace_name']}

    # Log: earliest log exceed ts for log.row_count with p95>0 and exceed_count_in_window>0
    log_candidate = pd.DataFrame()
    if (not log_groups.empty):
        log_candidate = log_groups[
            (log_groups['cmdb_id']==svc) &
            (log_groups['log_name']=='log.row_count') &
            (log_groups['p95'] > 0) &
            (log_groups['exceed_count_in_window'] > 0)
        ].copy()
    if log_candidate.empty:
        log_fields = {'earliest_log_exceed_ts': None, 'log_name': None}
    else:
        log_candidate = log_candidate.sort_values('earliest_exceed_dt')
        l = log_candidate.iloc[0]
        log_fields = {'earliest_log_exceed_ts': pd.to_datetime(l['earliest_exceed_ts'], utc=True), 'log_name': l['log_name']}

    # Compose row; keep datetimes as tz-aware or None
    row = {
        'cmdb_id': svc,
        'earliest_metric_anom_ts': metric_fields['earliest_metric_anom_ts'],
        'metric_kpi_name': metric_fields['metric_kpi_name'],
        'metric_run_length': metric_fields['metric_run_length'],
        'metric_run_max': metric_fields['metric_run_max'],
        'metric_global_p95': metric_fields['metric_global_p95'],
        'metric_severity_ratio': metric_fields['metric_severity_ratio'],
        'earliest_trace_exceed_ts': trace_fields['earliest_trace_exceed_ts'],
        'trace_name': trace_fields['trace_name'],
        'earliest_log_exceed_ts': log_fields['earliest_log_exceed_ts'],
        'log_name': log_fields['log_name'],
    }
    rows.append(row)

summary_df = pd.DataFrame(rows)

# For display, format datetime columns to ISO strings with +00:00 or None
for col in ['earliest_metric_anom_ts','earliest_trace_exceed_ts','earliest_log_exceed_ts']:
    summary_df[col] = summary_df[col].apply(lambda x: x.isoformat() if (pd.notna(x) and x is not None) else None)

# Determine earliest non-None timestamp across all metric/trace/log columns and service
def earliest_ts_for_row(r):
    times = []
    for c in ['earliest_metric_anom_ts','earliest_trace_exceed_ts','earliest_log_exceed_ts']:
        if r[c] is not None:
            times.append(pd.to_datetime(r[c], utc=True))
    return min(times) if times else None

summary_df['_earliest_any'] = summary_df.apply(earliest_ts_for_row, axis=1)
# Find service with global earliest timestamp
valid_earlies = summary_df[summary_df['_earliest_any'].notna()].copy()
if not valid_earlies.empty:
    idx = valid_earlies['_earliest_any'].idxmin()
    earliest_service = summary_df.loc[idx, 'cmdb_id']
    earliest_ts = valid_earlies.loc[idx, '_earliest_any'].tz_convert('UTC').isoformat()
    summary_line = f"Earliest non-None timestamp across all services: {earliest_service} at {earliest_ts}"
else:
    summary_line = "No non-None timestamps found across metric/trace/log for these services."

# Keep final display compact: drop helper column
summary_df = summary_df.drop(columns=['_earliest_any'])

# Also prepare up to 5 sample trace/log/metric rows? The user asked only summary table and short summary line.
# Return summary_df and summary_line
summary_df, summary_line

```
Out[7]:
```
Summary (one row per service):

- checkoutservice  
  - earliest_metric_anom_ts: 2024-01-16T10:14:00+00:00 (kpi = latency-50; run_length=1 min; run_max=0.14778; global_p95=0.13872; severity_ratio≈1.065)  
  - earliest_trace_exceed_ts: 2024-01-16T10:19:00+00:00 (trace.from_checkoutservice.row_count)  
  - earliest_log_exceed_ts: 2024-01-16T10:19:00+00:00 (log.row_count)

- currencyservice  
  - earliest_metric_anom_ts: 2024-01-16T10:07:00+00:00 (kpi = socket; run_length=25 min; run_max=3.63333; global_p95=3.00000; severity_ratio≈1.211)  
  - earliest_trace_exceed_ts: 2024-01-16T10:19:00+00:00 (trace.from_checkoutservice.row_count)  
  - earliest_log_exceed_ts: 2024-01-16T10:11:00+00:00 (log.row_count)

- emailservice  
  - earliest_metric_anom_ts: 2024-01-16T10:19:00+00:00 (kpi = workload; run_length=2 min; run_max=1.25987; global_p95=1.10568; severity_ratio≈1.139)  
  - earliest_trace_exceed_ts: 2024-01-16T10:19:00+00:00 (trace.from_checkoutservice.row_count)  
  - earliest_log_exceed_ts: 2024-01-16T10:19:00+00:00 (log.row_count)

- productcatalogservice  
  - earliest_metric_anom_ts: 2024-01-16T10:12:00+00:00 (kpi = workload; run_length=2 min; run_max=153.53778; global_p95=145.15211; severity_ratio≈1.058)  
  - earliest_trace_exceed_ts: 2024-01-16T10:08:00+00:00 (trace.from_recommendationservice.row_count)  
  - earliest_log_exceed_ts: None (no log.row_count exceedance with p95>0 in window)

- recommendationservice  
  - earliest_metric_anom_ts: 2024-01-16T10:13:00+00:00 (kpi = workload; run_length=1 min; run_max=20.17568; global_p95=19.03874; severity_ratio≈1.060)  
  - earliest_trace_exceed_ts: 2024-01-16T10:08:00+00:00 (trace.from_recommendationservice.row_count)  
  - earliest_log_exceed_ts: 2024-01-16T10:08:00+00:00 (log.row_count)

Earliest non-None timestamp across all metric/trace/log entries: currencyservice at 2024-01-16T10:07:00+00:00.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id    earliest_metric_anom_ts metric_kpi_name  metric_run_length  metric_run_max  metric_global_p95  metric_severity_ratio   earliest_trace_exceed_ts                                  trace_name     earliest_log_exceed_ts       log_name
0        checkoutservice  2024-01-16T10:14:00+00:00      latency-50                  1        0.147779           0.138724               1.065273  2024-01-16T10:19:00+00:00        trace.from_checkoutservice.row_count  2024-01-16T10:19:00+00:00  log.row_count
1        currencyservice  2024-01-16T10:07:00+00:00          socket                 25        3.633333           3.000000               1.211111  2024-01-16T10:19:00+00:00        trace.from_checkoutservice.row_count  2024-01-16T10:11:00+00:00  log.row_count
2           emailservice  2024-01-16T10:19:00+00:00        workload                  2        1.259867           1.105677               1.139453  2024-01-16T10:19:00+00:00        trace.from_checkoutservice.row_count  2024-01-16T10:19:00+00:00  log.row_count
3  productcatalogservice  2024-01-16T10:12:00+00:00        workload                  2      153.537783         145.152113               1.057772  2024-01-16T10:08:00+00:00  trace.from_recommendationservice.row_count                       None           None
4  recommendationservice  2024-01-16T10:13:00+00:00        workload                  1       20.175683          19.038737               1.059718  2024-01-16T10:08:00+00:00  trace.from_recommendationservice.row_count  2024-01-16T10:08:00+00:00  log.row_count, 'Earliest non-None timestamp across all services: currencyservice at 2024-01-16T10:07:00+00:00')```
```